# Databricks data ingestion
There are three methods that could be used to ingest data from files stored in the cloud storage.
- CREATE TABLE AS (CTAS) : 
    - It creates a delta table by default from files stored in the cloud object storage.
    - The ```read_files()``` function is used to read files from a specified loaction and return the data in a tabular format.
    - It offers several capabilities:
        - Supports various file formats like JSON, csv, xml, text, binaryfilem paraquet, avro and orc
        - Automatically detects file format and infers a unified schema across all files.
        - Allows you to specify format-specific options for greater control when reading source files.
        - Can be used in streaming tables to incrementally ingest files into delta lake using auto loader. We will learn more about auto loader shortly. 
- COPY INTO : 
    - This is used to copy files from cloud storage into the delta table. 
    - This command performs bulk load from files in cloud object storage into the table, and in this example, it will load files into the empty table new_table.
    - The FROM clause specifies the location of the csv files
    - You start by creating a table which can be defined with or without a schema. In this case, we will create a table named new_table without a schema.
    - COPY INTO is ideal for situations where the cloud storage location is continuously adding files, since it is a reltriable and indepodent operation designed for incremental batch ingestion.
        - What that means is : COPY INTO will skip any files that have already been loaded into the table, and only new files will be ingested. 
        - Now lets go over some of the key aspects of COPY INTO: 
            - It supports various common file types liek parquet, JSON, XML and others.
            - The FROM clause specifies the path of the cloud storage where new files are being continously added.
            - FORMAT_OPTIONS{} controls how the source files are parsed and interpreted and the available options will depend on the file format you are working with.
            - COPY_OPTIONS() lets you control the behaviour of COPY INTO operation itself. For example: options like schema evolution using mergeSchema, or idempotency using force.
- Auto Loader : 
    - Incrementally and efficiently process new data files as they arrive in cloud storage without any additional setup.
    - Auto loader has support for both python and sql (leveraging declarative pipelines)
    - You can use Auto Loader to process billions of files. 
    - Auto loader is built upon **Spark Structured Streaming**

## Schema : 
- A schema (in terms of databases) is a formal language which describes the structure of data (blueprint) of a database.
- A schema can define many different data structures that serve different purpose for a database.
- Different data structures (relational databases):
    - Tables
    - Fields
    - Views
    - Relationships
    - Indexes
    - Packages
    - Procedures
    - Functions
    - XML schemas
    - Queues
    - Triggers
    - Types
    - Sequences
    - materialized views
    - Synonyms
    - database links
    - Directories
## Schemaless : 
- Schemaless is when the primary "cell" of database can accept many types.
- This allows developers to forgo the upfront data modelling.
- Common schemaless databases are  : 
    - Key/Value
    - Document
    - Columns
        - Wide column
    - graph

## Data Documents
- A data document defines the collective form in which data exists.
- Common types of data documents : 
    - Datasets : a logical grouping of data
    - Databases : structured data that can be quickly accessed and searched
    - Datastores : unstructured or semi-structured data to housing data
    - Data warehouse : structured or semi-structured data for creating reports and analytics.
    - Notebooks : data that is arranged in pages, designed for easy consumption

## Data sets
- A data sets is a logical grouping of units of data that generally are closely related and/or share the same data structure.
- Just because I said data structure doesn't always mean that the data itself is structured, it can be a semi-structured or un-structured data
- There are publically available data sets that are used in the learning of statistics, data analytics, machine learning
- MNIST database Images of handwritten digits used to test classification, clustering and image processing algorithms.
- Commonly used when learning how to build computer vision ML models to translate handwriting into digital text.
- COCO dataset (Common objects in Context dataset) : A dataset which contains many common images using a JSON file (coco format) that identify objects or segments within an image.
- IMDB riviews datasets : A movie dataset with 25,000 highly popular movie reviews for training and 25000 for testing.


## Query and Querying 
- A query is a request for data results (reads) or to perform operations such as inserting updating deleting data (writes).
- A query can perform maintanance operations on the data and is not always restricted to just working with the data that resides the database.
- Querying : This is an act of performing a query
- what is a query language ? : A scripting language designed as the format to submit a request or actions to the database. Notable query languages: 
    - SQL
    - GraphSQL
    - Kusto
    - Xpath
    - Gremlin

## Batch vs Stream processing
### Batch processing
![batch_processing](images/batch_processing.png)
- When you send batches (a collection) of data to be processed. 
- Batches are generally scheduled example : Every day at 1PM
- Batches are not real-time.
- Batche processing is ideal for very large processing workloads.
- Batch processing is more cost-effective
### Stream processing
![stream_processing](images/stream_processing.png)
When you process data as soon as it arrives : 
- Produces will send data to a stream 
- Consumers will pull from the stream
- A data can be held in a stream for a period of time so that we can have a better re-usability of data.
- It is suitable for real-time processing (streaming-video)
- Much more expensive than batch processing.


## Pivot table
- A pivot table is a table of statistics that summarizes the data of a more extensive table from a : Database, Spreadsheet or Business intelligence (BI) tool
- Pivot tables are a technique in data processing.
- They arrange and rearrange (or "Pivot") statistics in order to draw attention to useful information
- This leads to finding figures and facts quickly making them integral to data analysis.
- Here is the example table :

    | Region | Product | Sales |
    | ------ | ------- | ----- |
    | East   | Apples  | 100   |
    | East   | Bananas | 150   |
    | West   | Apples  | 200   |
    | West   | Bananas | 120   |

- Here is the pivot table of the above table : 

    | Region    | Apples  | Bananas | Total   |
    | --------- | ------- | ------- | ------- |
    | East      | 100     | 150     | 250     |
    | West      | 200     | 120     | 320     |
    | **Total** | **300** | **270** | **570** |

- Here is a sample code for generating a pivot table from a table (python > Pandas)
    ```python
    import pandas as pd

    data = {
        "Region": ["East", "East", "West", "West"],
        "Product": ["Apples", "Bananas", "Apples", "Bananas"],
        "Sales": [100, 150, 200, 120],
    }
    df = pd.DataFrame(data)

    pivot = df.pivot_table(index="Region", columns="Product", values="Sales", aggfunc="sum")
    print(pivot)
    ```
- Here is a sample code for generating a pivot table from a table in (pyspark)
    ```python
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import sum

    spark = SparkSession.builder.getOrCreate()

    data = [("East", "Apples", 100),
            ("East", "Bananas", 150),
            ("West", "Apples", 200),
            ("West", "Bananas", 120)]

    df = spark.createDataFrame(data, ["Region", "Product", "Sales"])

    pivot_df = df.groupBy("Region").pivot("Product").agg(sum("Sales"))
    pivot_df.show()
    ```
- **When to use pivot table:**
    - Summarize large datasets quickly
    - Find totals, averages or counts grouped by categories.
    - compare data across multiple dimensions (example region vs products)
    - Explore patterns or trends in your dataset.
    


## Relational data 
### Tables : 
- A logical grouping of rows and columns. Think like a Excel spreadsheet.
- Tabular data --- data that makes use of table data strucutres
### Views : 
- Views is a result set of a stored query on data stored in memory (a temporary or virtual table)
### Materialized Views : 
- Material Views is a result set of stored query on data stored on disk.
### Indexes : 
- A copy of your data sorted by one or multiple columns for faster reads at cost of storage.
### Constraints : 
- Rules applied to writes, that can ensure data integrity example : don't allow duplicate records.
### Triggers : 
- A function that is triggered on specific database events.
### Primary key : 
- One or multiple columns that uniquely identify a table in a row
### Foreign key :
- A column which holds the value of primary key from another key to establish a relationship.
    - A relationship is when two tables have a reference to one another to join data togeather.

## Relational data -- Relationships
- A relational database establish connection to other tables via foreign keys referencing another table's primary key.
- Types of relation : 
    - **One to one** : A country has a capital
    - **One to many** : A Store has many customers
    - **Many to many** : A project has many tasks and Tasks can belong to many projects.
    - **Many to Many (via Join/Junction table)** : A student has many classes through enrollments. A class has many students through enrollments.
    

## Indexes
- A database index is a data structure that improves the speed of reads from the database table by storing the same or partial redundant data organized in a more efficient logical order.
- A logical order is commonly determined by one or more sort ke(s)
- A common data structure of an index is a balanced Tree (B-Tree)

## Non-relational data
- A non-relational database stores data in a non-tabular form and will be optimized for different kinds of data-strucutres.
- Types of non-relational databases:
    - **Key/value**
        - Each value has a key
        - Designed to scale
        - Only simple lookups
    - **Document**
        - Primary entity is a JSON-like data-structure called a document.
    - **Columnar**
        - Has a table-like structure but data is stored around columns instead of rows.
    - **GRAPH**
        - Data is represented with nodes and structures. Where relationships matter
        

## Data integrity and Data corruption
### Data integrity
- Data integrity is the maintenance and assurance of data accuracy and consitency over its entire life-cycle.
- Its used as proxy term for data quality, data validation is a pre-requisite for data integrity.
- The goal of data integrity ensure data is recorded exactly as intended.
### Data corruption
- Data corruption is the act or state of data not being in the intended state and will result in data or misinformation.
- Data corruption occurs when unintended changes result when reading and writing:
    - Unexpected hardware failure.
    - Human error when inputing or modifying data
    - Malicious actors with intent of corrupting your data.
    - Unforeseen side effects for automated operations via computer code.
### Ways to insure data integrity:
- Have a well defined and documented data modelling.
- Logical constraints on your database items.
- Redundant and versions of your data to compare and restore
- Human analysis of the data
- Hash functions to determine if changes have been tampered

## Normalized vs De-normalized data
### Normalized 
A schema design to store non-redundant and consistent data.
- Data integrity is maintained
- Little to no redundant data
- Many tables
- Optimizes for storage of data
### Denormalized
A schema that combines data so that accessing data (querying) is fast.
- data integrity is not maintained
- Redundant data is common
- Fewer tables
- Excessive data, storage is less optimal

## Strongly consistent vs Eventually consistent
### **What is data consistency?**
When data being kept in two different place and whether the data exactly match or do not match.

When you have to have duplicate your data in many places and need to keep them up to date to be exact matching, based on how data is transmitted and service levels cloud service

#### Strongly consistent
Every time you request data (query) you can expect consistent data to be returned with x time (1 seconds)

We will never return to you with old data. But you have to wait at least 2 seconds for the query to return.

#### Eventually consistent
When you request data you may get back inconsistent data within 2 seconds.

We are giving you whatever data is currently in the database you may get new data or old data but if you wait a little bit longer it will generally be up to date.

## Synchronous vs Asynchronous
### Synchronous 
Continuous stream of data that is synchronized by a timer or clock (guarantee of time). Can only access data once transfer is complete.
- Guarantee consitency of data return at time of access
- Slower access times
### Asynchronous
Continuous stream of data seperated by start and stop bits (no guarantee of time)
Can access data anytime but may return older version or empty
placeholder
- Faster access times not guarantee of consistency


## Data source 
A data source is where data originates from. An analytics tool may be connected to various data sources to create a visualization or report.

A data source could be a : 
- Data Lake 
- Data Warehouse
- Datastore
- Database
- Data requested on demand from an API endpoint from a web app
- Flat files (example excel spreadsheet)
## Data Store
- A data store is a repository for persistently storing and managing collections of unstructured or semi-structured data.
- A database is a sub-set of a data store.
- It is generally a data store indicates working unstructured or semi-structured data.
- A datastore can be specialized in storing : 
    - Relational databases
    - NoSQL databases
    - Object oriented databases
- Data stores are designed to be distributed across many machines.
- Directory service
## Database 
- A database is a data-store that stores semi-structred and structured data.
- A database is more complex data stores because it requires using formal design and modeling technoques.
- Database can be generally categorized as either:
    - Relational databases : 
        - Structured data that strongly represents tabular data (tables, rows and columns) 
        - Row oriented or columnar oriented
    - Non-relational databases : 
        - Semi-structured that may or may not distantly resemble tabular data.
- Databases have a rich set of functionality
    - specialized language to query (retrieve data) 
    - specialized modeling stratagies to optimize retrieval for different use cases
    - more fine tune control over the transformation of the data into useful data structures or reports
## Data warehouse 
- A relational datastore designed for analytics workloads, which generally column-oriented data-store
- Companies will have terabytes of rows of data and they need a fast way to be able to produce analytics reports.
- Data warehouses generally perform aggregation.
- aggregation is grouping data example find the total or average
- data warehouses are optimized around columns since they need to quickly aggregate column data.
- Data warehouses are generally designed to be HOT
- HOT means they can returned queries very very fast even though they have vast amounts of data.
Data warehouses are infrequently accessed meaning they aren't intended for real-time reporting but maybe once or twice a day or once a week to generate business and user reports.
- A data warehouse needs to consume data from a relational databases on a regular basis.
- Generally datawarehouses are read-only where the data is only read and we normally don't use it for transactional data.
## Data Mart
- A data mart subset of a data warehouse
- A data mart will store data under 100 GB and has a single business focus
- Data mart allows different teams or departments to have control over their own dataset for their specific use case.
- Data marts are generally designed to be read-only
- Data marts also increase the frequency at which data can be accessed.
- The cost to query the data is much lower and so queries can be performed multiple times a day or even hourly.
